# 03 — Datasets and augmentations

**Day 1 · Notebook 3 of 4**

By the end you can:

1. Load a torchvision dataset and iterate over batches with `DataLoader`.
2. Apply train-time augmentations and visualize them.
3. Spot the difference between "transform applied to the image" and
   "transform applied to the model input."


In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms as T
import matplotlib.pyplot as plt

from cvcourse import show_grid

## 1. A `Dataset` is just an indexable thing

Anything that supports `len()` and `dataset[i]` works. torchvision ships many.


In [ ]:
train_ds = torchvision.datasets.FashionMNIST(
    root="../data", train=True, download=True,
    transform=T.ToTensor(),
)
test_ds = torchvision.datasets.FashionMNIST(
    root="../data", train=False, download=True,
    transform=T.ToTensor(),
)
print("train:", len(train_ds), "test:", len(test_ds))
img, label = train_ds[0]
print("first sample:", img.shape, img.dtype, "label:", train_ds.classes[label])

## 2. `DataLoader` = batching + shuffling + parallel workers


In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
xb, yb = next(iter(train_loader))
print("batch images:", xb.shape, "labels:", yb.shape)
show_grid(xb[:16], titles=[train_ds.classes[i] for i in yb[:16]], cols=8)

## 3. Augmentation: same image, more views

Augmentations make the model see each training image many ways. Use them at
**train time only** — never on the validation/test set.


In [ ]:
aug = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ToTensor(),
])

# Reload one sample with augmentation, show 8 random versions of the same image.
train_aug = torchvision.datasets.FashionMNIST(
    root="../data", train=True, download=False, transform=aug,
)
versions = [train_aug[0][0] for _ in range(8)]
show_grid(versions, cols=8)

### Pitfall: don't augment your validation set

If you augment validation, the metric becomes noisy and you can't compare
models. Keep val/test transforms deterministic — usually just resize +
normalize.


## 4. Normalization belongs in the transform pipeline


In [ ]:
pipeline = T.Compose([
    T.ToTensor(),
    T.Normalize((0.2860,), (0.3530,)),   # FashionMNIST mean/std
])
train_ds_norm = torchvision.datasets.FashionMNIST(
    root="../data", train=True, download=False, transform=pipeline,
)
img, _ = train_ds_norm[0]
print("after normalize -> mean:", img.mean().item(), "std:", img.std().item())

## Recap

- `Dataset` + `DataLoader` is the standard pattern.
- Augment training data; keep validation deterministic.
- Normalize inside the transform pipeline so it's part of the data, not the model.

Next: **first CNN** — putting it all together.
